[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kevisback/bda-course/blob/main/wise2627/notebooks/09_wann_das_llm_luegt.ipynb)

# Sitzung 9 — Wann das LLM lügt

**Big Data Analytics (W3-BDA) · HTW Berlin · Master WI · 02.12.2026**

Letzte Woche stand am Ende ein fertiges Dashboard mit einem Absatz für den Chef. Der Absatz klang gut. Heute stellen wir die unbequeme Frage: **Stimmt er?**

Das Gefährliche an einer falschen Zusammenfassung ist nicht, dass sie falsch ist. Es ist, dass sie **genauso überzeugend klingt** wie eine richtige, und dass der Chef nie die Rohdaten sieht, mit denen er sie vergleichen könnte.

**Ablauf**
1. **Der Absatz aus Sitzung 8 (Abschnitt 1):** Ausgangspunkt, und warum er sich gut anfühlt.
2. **Zusammenfassung aus Rohdaten (Abschnitt 2):** Was passiert, wenn das LLM selbst zählen soll?
3. **Satz für Satz prüfen (Abschnitt 3):** ein Werkzeug, das jede Aussage einer Quelle zuordnet.
4. **Erfundene Themen (Abschnitt 4):** Kritikpunkte, die in keiner Bewertung stehen.
5. **Warum lügt ein LLM? (Abschnitt 5):** der Mechanismus, und warum dieselbe Frage verschiedene „Fakten" liefert.
6. **Fehlklassifikationen diagnostizieren (Abschnitt 6):** Ironie, Sprache, und die Begründung als Spur.
7. **Gegenmittel (Abschnitt 7):** Was hilft, und was nicht.

Heute brauchen wir den **API-Schlüssel**. Die Ausgaben des LLM sind bei euch etwas anders als bei mir, und genau darum geht es heute.

### Rückblick auf Sitzung 8

```text
1. Pipeline                  -> Bausteine als Funktionen: bereinigen, analysieren, prüfen,
                                Kennzahlen, Dashboard, Zusammenfassung
2. Analysieren               -> Schema + Korrektur (Sitzung 6), parallel + Retry (Sitzung 3)
3. Prüfen                    -> einzeln (Treffer, F1) UND im Aggregat: ändert sich die Antwort?
4. Dashboard                 -> vier Fragen, vier Felder; jede Zahl mit ihrer Einschränkung
5. Zusammenfassung           -> LLM formuliert aus geprüften Fakten, rechnet nicht selbst
6. Zahlen-Check und Belege   -> jede Zahl und jede Behauptung auf eine Quelle zurückführen
```

Ihr habt eine Notiz aus dem Plausibilitätscheck mitgebracht: Welcher Satz war am schwersten zu prüfen? Heute sehen wir, warum.

## 0. Setup und die Pipeline aus Sitzung 8

**Was passiert in der nächsten Zelle?** Bibliotheken, `radar.py`, Modell, Daten, wie immer.

In [ ]:
!pip install anthropic --quiet

import json, re, time, random, urllib.request, importlib
from concurrent.futures import ThreadPoolExecutor
from typing import Literal
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown
from pydantic import BaseModel, Field, ValidationError
pd.set_option("display.max_colwidth", 90)

URL = "https://raw.githubusercontent.com/kevisback/bda-course/main/wise2627/notebooks/radar.py"
try:
    urllib.request.urlretrieve(URL, "radar.py")
except Exception as e:
    print("Download fehlgeschlagen, nutze lokale radar.py:", e)
import radar
importlib.reload(radar)
radar.setup(backend="anthropic", model="claude-haiku-4-5")

df, wahrheit = radar.lade_sauber()
THEMEN = radar.THEMEN
REIHENFOLGE = ["positive", "mixed", "neutral", "negative"]
print(df.shape)

Heute bauen wir nichts Neues an der Pipeline. Die nächsten drei Zellen sind **Sitzung 8 in kompakt**: dieselben Funktionen, ohne die Erklärungen. Wer nachlesen will, findet alles in Notebook 08.

```text
analysiere(text)      -> Stimmung, Themen, Begründung, dringend   (Schema + Korrektur, Sitzung 6)
kennzahlen(teil)      -> alle Zahlen für einen Ausschnitt        (Sitzung 7 und 8)
fakten(k, titel)      -> die Zahlen als kurze Liste fürs LLM      (Sitzung 8)
zusammenfassung(f)    -> der Absatz für den Chef                  (Sitzung 8)
zahlen_check, belege  -> die Prüfwerkzeuge                        (Sitzung 8)
```

**Was passiert in der nächsten Zelle?** Schema, Prompt und `analysiere`, wie in Sitzung 8.

In [ ]:
class Nennung(BaseModel):
    thema: Literal[tuple(THEMEN)]
    polaritaet: Literal["positiv", "negativ"]

class Analyse(BaseModel):
    sentiment: Literal["positive", "negative", "mixed", "neutral"]
    themen: list[Nennung]
    begruendung: str = Field(max_length=200)
    dringend: bool

ANALYSE_PROMPT = """Du analysierst eine Produktbewertung zu kabellosen Ohrhörern.

Antworte ausschließlich mit einem JSON-Objekt mit genau diesen Feldern:
- "sentiment": "positive", "negative", "mixed" oder "neutral"
- "themen": Liste von {"thema": <Thema>, "polaritaet": "positiv" oder "negativ"}
   Erlaubte Themen: """ + ", ".join(THEMEN) + """
- "begruendung": höchstens 20 Wörter
- "dringend": true, wenn ein Defekt oder ein Problem gemeldet wird, um das sich jemand kümmern sollte, sonst false"""

def pruefe(roh):
    m = re.search(r"\{.*\}", roh or "", re.DOTALL)
    if not m:
        return None, "Kein JSON-Objekt gefunden."
    text = re.sub(r",\s*([}\]])", r"\1", m.group(0))
    try:
        daten = json.loads(text)
    except json.JSONDecodeError as e:
        return None, f"Kein gültiges JSON: {e.msg}"
    try:
        return Analyse.model_validate(daten), None
    except ValidationError as e:
        return None, "; ".join(f"{'.'.join(str(x) for x in f['loc'])}: {f['msg']}" for f in e.errors())

def frage_modell(text, korrektur=None):
    user = f'Bewertung:\n"""{text}"""'
    if korrektur:
        alt, fehler = korrektur
        user += (f"\n\nDeine letzte Antwort war ungültig.\nAntwort: {alt}\nFehler: {fehler}\n"
                 "Antworte nur mit dem korrigierten JSON-Objekt.")
    if radar.BACKEND == "mock":
        themen = radar._mock_themen(text)
        s = json.loads(radar.call_llm(user, temperature=0)["text"])["sentiment"]
        return json.dumps({"sentiment": s, "themen": themen, "begruendung": "Mock: Wortsignale",
                           "dringend": any(t["polaritaet"] == "negativ" for t in themen)}, ensure_ascii=False)
    return radar.mit_retry(radar.call_llm, user, system=ANALYSE_PROMPT, temperature=0, max_tokens=400)["text"]

def analysiere(text, max_korrekturen=2):
    roh = frage_modell(text)
    for anfrage in range(1, max_korrekturen + 2):
        analyse, fehler = pruefe(roh)
        if analyse:
            return analyse, anfrage
        if anfrage <= max_korrekturen:
            roh = frage_modell(text, korrektur=(roh, fehler))
    return None, anfrage

**Was passiert in der nächsten Zelle?** Alle 279 Bewertungen analysieren, acht gleichzeitig, und die Ergebnisse in Spalten auspacken. Dazu die Wahrheit aus dem Generator, inklusive zwei Merkmalen, die wir heute brauchen: Ist eine Bewertung **ironisch**? Ist sie **englisch**?

In [ ]:
radar.reset_verbrauch()
start = time.time()
with ThreadPoolExecutor(max_workers=8) as pool:
    ergebnisse = list(pool.map(analysiere, df["text"]))

df["analyse"] = [a for a, _ in ergebnisse]
ok = df["analyse"].notna()
df["sentiment_llm"] = [a.sentiment if a else None for a in df["analyse"]]
df["themen_llm"] = [[(n.thema, n.polaritaet) for n in a.themen] if a else [] for a in df["analyse"]]
df["dringend"] = [a.dringend if a else None for a in df["analyse"]]
df["begruendung"] = [a.begruendung if a else None for a in df["analyse"]]

w = wahrheit.loc[df["review_id"]]
df["sentiment_wahr"] = w["true_sentiment"].values
df["ironisch"] = w["is_sarcastic"].values
df["englisch"] = w["is_english"].values
df["quartal"] = df["datum"].dt.to_period("Q")

print(f"{ok.sum()} von {len(df)} analysiert in {time.time() - start:.0f} s | {radar.verbrauch()}")

**Was passiert in der nächsten Zelle?** Kennzahlen, Fakten, Zusammenfassung und die zwei Prüfwerkzeuge aus Sitzung 8. Neu ist nur eine kleine Hilfsfunktion `frage_llm`, die wir heute für alle freien Textanfragen benutzen.

In [ ]:
def frage_llm(user, system=None, temperature=0, max_tokens=600, mock=""):
    """Eine freie Textanfrage ans LLM. Im Mock kommt der Text aus `mock` zurück."""
    if radar.BACKEND == "mock":
        return "(Mock) " + mock
    return radar.mit_retry(radar.call_llm, user, system=system,
                           temperature=temperature, max_tokens=max_tokens)["text"].strip()

def kennzahlen(teil):
    g = teil[teil["analyse"].notna()]
    k_neg = int((g["sentiment_llm"] == "negative").sum())
    quartale = (g.dropna(subset=["datum"]).assign(neg=lambda x: x["sentiment_llm"] == "negative")
                 .groupby("quartal")["neg"].agg(["sum", "count"]))
    nennungen = pd.DataFrame([{"thema": t, "polaritaet": p} for l in g["themen_llm"] for t, p in l],
                             columns=["thema", "polaritaet"])
    return {"bewertungen": len(teil), "analysiert": len(g), "aufgegeben": len(teil) - len(g),
            "zeitraum": f"{g['datum'].min():%d.%m.%Y} bis {g['datum'].max():%d.%m.%Y}",
            "anteil_negativ": k_neg / len(g), "intervall_negativ": radar.wilson(k_neg, len(g)),
            "quartale": quartale,
            "kritik": nennungen[nennungen["polaritaet"] == "negativ"]["thema"].value_counts(),
            "lob": nennungen[nennungen["polaritaet"] == "positiv"]["thema"].value_counts(),
            "anteil_dringend": g["dringend"].mean(),
            "sterne_mittel": g["sterne"].astype("float").mean(), "sterne_fehlt": g["sterne"].isna().mean()}

def fakten(k, titel):
    q = k["quartale"]
    return {"titel": titel, "zeitraum": k["zeitraum"],
            "bewertungen_ausgewertet": k["analysiert"], "nicht_auswertbar": k["aufgegeben"],
            "anteil_negativ_prozent": round(100 * k["anteil_negativ"]),
            "intervall_negativ_prozent": [round(100 * x) for x in k["intervall_negativ"]],
            "anteil_negativ_pro_quartal_prozent": {str(i): round(100 * s / n) for i, s, n in zip(q.index, q["sum"], q["count"])},
            "bewertungen_pro_quartal": {str(i): int(n) for i, n in zip(q.index, q["count"])},
            "haeufigste_kritik_nennungen": k["kritik"].head(3).to_dict(),
            "haeufigstes_lob_nennungen": k["lob"].head(3).to_dict(),
            "anteil_dringend_prozent": round(100 * k["anteil_dringend"]),
            "sterne_mittel": round(k["sterne_mittel"], 1),
            "anteil_ohne_sterne_prozent": round(100 * k["sterne_fehlt"])}

SUMMARY_PROMPT = """Du schreibst für eine Produktmanagerin eine Zusammenfassung von Kundenbewertungen.

Regeln:
- Verwende ausschließlich die Zahlen aus den Fakten. Rechne nichts Neues aus, erfinde nichts dazu.
- 4 bis 5 Sätze, sachlich, auf Deutsch, ohne Aufzählungszeichen.
- Nenne den Trend, die wichtigsten Kritikpunkte und was gut läuft.
- Nenne mindestens eine Einschränkung (z. B. Unsicherheit, fehlende Sterne, nicht auswertbare Bewertungen).
- Keine Vermutungen über Ursachen."""

def zusammenfassung(f, temperature=0):
    kritik = ", ".join(f["haeufigste_kritik_nennungen"])
    return frage_llm("Fakten:\n" + json.dumps(f, ensure_ascii=False, indent=1), system=SUMMARY_PROMPT,
                     temperature=temperature, max_tokens=500,
                     mock=f"Von {f['bewertungen_ausgewertet']} Bewertungen sind {f['anteil_negativ_prozent']} % negativ. "
                          f"Am häufigsten kritisiert werden {kritik}.")

def erlaubte_zahlen(f):
    erlaubt = set()
    def sammle(x):
        if isinstance(x, dict):
            for v in x.values(): sammle(v)
        elif isinstance(x, (list, tuple)):
            for v in x: sammle(v)
        elif isinstance(x, (int, float, np.integer, np.floating)):
            erlaubt.add(round(float(x), 1))
        elif isinstance(x, str):
            for z in re.findall(r"\d+(?:[.,]\d+)?", x):
                erlaubt.add(round(float(z.replace(",", ".")), 1))
    sammle(f)
    return erlaubt

def zahlen_check(text, f):
    erlaubt = erlaubte_zahlen(f)
    return pd.DataFrame([{"Zahl im Text": z,
                          "in den Fakten?": "✅" if round(float(z.replace(",", ".")), 1) in erlaubt else "❓ prüfen"}
                         for z in re.findall(r"\d+(?:[.,]\d+)?", text)])

def belege(thema, polaritaet="negativ", n=4, teil=None):
    teil = df if teil is None else teil
    maske = [(thema, polaritaet) in l for l in teil["themen_llm"]]
    treffer = teil[maske][["review_id", "text", "begruendung"]].drop_duplicates("text")
    print(f"{sum(maske)} Bewertungen mit {thema} ({polaritaet}), {min(n, len(treffer))} davon:")
    with pd.option_context("display.max_colwidth", None):
        display(treffer.head(n))

k_alle = kennzahlen(df)
f_alle = fakten(k_alle, "Nimbus Q2, alle Bewertungen")
print("Fakten bereit:", list(f_alle))

## 1. Der Absatz aus Sitzung 8

**Was passiert in der nächsten Zelle?** Die Zusammenfassung aus geprüften Fakten, wie letzte Woche, und der Zahlen-Check dazu.

In [ ]:
text_fakten = zusammenfassung(f_alle)
display(Markdown("**Zusammenfassung aus Fakten:**\n\n" + text_fakten))
zahlen_check(text_fakten, f_alle)

**Wie lese ich das?** Ein sauberer Absatz, die Zahlen stehen in den Fakten. Das ist der **beste Fall**: Das LLM bekommt nur geprüfte Zahlen und soll nur formulieren.

In der Praxis läuft es oft anders. Jemand hat keine Pipeline, nur einen Stapel Bewertungen und ein Chatfenster: *„Fass mir das mal zusammen."* Genau das probieren wir jetzt.

## 2. Zusammenfassung aus Rohdaten

Statt Fakten bekommt das LLM jetzt **40 Bewertungen**, so wie man sie in ein Chatfenster kopieren würde, und soll selbst zusammenfassen, mit Prozentangaben.

**Was passiert in der nächsten Zelle?** 40 zufällige Bewertungen, ein freundlicher Prompt ohne strenge Regeln, und die Antwort.

In [ ]:
stichprobe = df.sample(40, random_state=9)
liste = "\n".join(f"- {t}" for t in stichprobe["text"])

ROH_PROMPT = "Du fasst Kundenbewertungen für eine Produktmanagerin zusammen."
roh_anfrage = (f"Hier sind Bewertungen zu den Ohrhörern Nimbus Q2:\n{liste}\n\n"
               "Fasse in 4 bis 5 Sätzen zusammen, wie zufrieden die Kunden sind und was sie am meisten stört. "
               "Nenne dabei Prozentangaben.")

mock_roh = "Rund 60 % der Kunden sind zufrieden, besonders mit dem Klang. Etwa 30 % beklagen den Akku, viele auch den Kundenservice."
text_roh = frage_llm(roh_anfrage, system=ROH_PROMPT, mock=mock_roh)
display(Markdown("**Zusammenfassung aus 40 Rohbewertungen:**\n\n" + text_roh))

**Wie lese ich das?** Wieder ein flüssiger, überzeugender Absatz, diesmal mit Prozentzahlen. Woher kommen die? Das LLM hat keine Tabelle und keinen Taschenrechner. Es hat 40 Zeilen Text gelesen und dann **geschrieben, was nach einer guten Zusammenfassung klingt**.

**Was passiert in der nächsten Zelle?** Wir vergleichen die Prozentzahlen im Text mit dem, was in den 40 Bewertungen wirklich steht (laut Wahrheit) und was die Pipeline für alle 279 Bewertungen gemessen hat.

In [ ]:
wahr_stichprobe = stichprobe["sentiment_wahr"].value_counts(normalize=True).reindex(REIHENFOLGE, fill_value=0)
pipeline_alle = df.loc[ok, "sentiment_llm"].value_counts(normalize=True).reindex(REIHENFOLGE, fill_value=0)
vergleich = pd.DataFrame({"in den 40 (wahr)": wahr_stichprobe, "alle 279 (Pipeline)": pipeline_alle})
display(vergleich.style.format("{:.0%}"))

print("Prozentangaben im Text:", re.findall(r"\d+(?:[.,]\d+)?\s*%", text_roh) or "keine")
print("\nZahlen-Check gegen die Fakten aller Bewertungen:")
zahlen_check(text_roh, f_alle)

**Wie lese ich das?** Vergleicht jede Prozentzahl im Text mit der Tabelle. Typisch sind drei Dinge:
- **Gerundete Schätzungen**, die ungefähr passen („rund 40 %"), aber nicht gezählt wurden.
- **Zahlen, die zu keiner Spalte passen.** Sie klingen präzise, sind aber eine Fortsetzung, die „richtig aussieht".
- **Aussagen über alle Kunden**, obwohl das LLM nur 40 Bewertungen gesehen hat. Von 40 auf „die Kunden" zu schließen, ist genau die Verallgemeinerung, vor der wir in Sitzung 3 gewarnt haben.

Der Zahlen-Check markiert fast alles mit ❓, und das ist richtig: Keine dieser Zahlen hat jemand gezählt.

> **Nicht:** „Das LLM hat die Bewertungen gelesen, also stimmen die Prozente."
> **Sondern:** Lesen ist nicht Zählen. Zahlen kommen aus der Pipeline, das LLM formuliert nur.

## 3. Satz für Satz prüfen

Der Zahlen-Check findet nur Zahlen. Eine Zusammenfassung besteht aber aus **Aussagen**: „Die Kunden lieben den Klang." „Der Akku ist das größte Problem." Für jede Aussage gibt es drei Möglichkeiten:

```text
belegt        -> steht so in den Fakten (Zahl stimmt, Thema an der richtigen Stelle)
prüfbar       -> könnte stimmen, muss man nachschlagen (Thema ohne Zahl -> belege(...))
nicht belegt  -> steht nirgends in den Fakten (erfundene Zahl, Thema, das nicht vorkommt)
```

**Was passiert in der nächsten Zelle?** Eine Funktion `satz_check`, die einen Text in Sätze zerlegt und für jeden Satz sagt, was sie findet: Zahlen, Themen aus dem Katalog, Themen außerhalb des Katalogs.

In [ ]:
STICHWORTE = {"Klang": ["klang", "sound", "bass", "audio"], "Geräuschunterdrückung": ["geräuschunterdrückung", "anc", "noise"],
              "Akku & Laden": ["akku", "lade", "batterie", "battery"], "Tragekomfort": ["tragekomfort", "passform", "sitz", "bequem"],
              "Verbindung": ["verbindung", "bluetooth", "connection"], "Verarbeitung & Haltbarkeit": ["verarbeitung", "haltbar", "case", "qualität"],
              "Preis": ["preis", "teuer", "price"], "App & Bedienung": ["app", "bedienung", "touch", "steuerung"]}
VERDAECHTIG = ["kundenservice", "service", "lieferung", "versand", "rückgabe", "garantie", "verpackung", "support"]

def satz_check(text, f):
    erlaubt = erlaubte_zahlen(f)
    kritik, lob = set(f["haeufigste_kritik_nennungen"]), set(f["haeufigstes_lob_nennungen"])
    zeilen = []
    for satz in re.split(r"(?<=[.!?])\s+", text.strip()):
        klein = satz.lower()
        zahlen = re.findall(r"\d+(?:[.,]\d+)?", satz)
        falsch = [z for z in zahlen if round(float(z.replace(",", ".")), 1) not in erlaubt]
        themen = [t for t, ws in STICHWORTE.items() if any(w in klein for w in ws)]
        fremd = [w for w in VERDAECHTIG if w in klein]
        if falsch or fremd:
            urteil = "❌ nicht belegt"
        elif themen and not zahlen:
            urteil = "🔎 prüfbar (belege)"
        elif zahlen:
            urteil = "✅ Zahlen belegt"
        else:
            urteil = "– keine prüfbare Aussage"
        zeilen.append({"Satz": satz, "Zahlen nicht in Fakten": ", ".join(falsch),
                       "Themen": ", ".join(t + (" (Kritik)" if t in kritik else " (Lob)" if t in lob else "") for t in themen),
                       "außerhalb des Katalogs": ", ".join(fremd), "Urteil": urteil})
    return pd.DataFrame(zeilen)

with pd.option_context("display.max_colwidth", None):
    display(satz_check(text_fakten, f_alle))

**Wie lese ich das?** Jeder Satz eine Zeile. Bei der Zusammenfassung aus Fakten sollten die meisten Sätze ✅ oder 🔎 sein. In der Spalte „Themen" steht in Klammern, ob das Thema in den Fakten bei der Kritik oder beim Lob vorkommt. Steht dort ein Thema ohne Klammer, nennt der Satz ein Thema, das in den Top 3 gar nicht vorkommt.

Die Funktion ist bewusst einfach, mit Stichwortlisten wie in Sitzung 5. Sie ersetzt das Lesen nicht, sie sagt euch, **wo** ihr lesen müsst.

**Was passiert in der nächsten Zelle?** Dasselbe für die Zusammenfassung aus Rohdaten.

In [ ]:
with pd.option_context("display.max_colwidth", None):
    display(satz_check(text_roh, f_alle))

**Wie lese ich das?** Hier sieht es meist anders aus: Zahlen, die nicht in den Fakten stehen, vielleicht ein Thema außerhalb unseres Katalogs. Jeder ❌-Satz wäre ungeprüft beim Chef gelandet, und er hätte keine Chance gehabt, den Fehler zu sehen.

Für die 🔎-Sätze nehmen wir `belege` aus Sitzung 8. **Was passiert in der nächsten Zelle?** Wir schauen nach, ob die häufigste Kritik aus den Fakten wirklich in den Bewertungen steht.

In [ ]:
belege(k_alle["kritik"].index[0])

## 4. Erfundene Themen

Besonders tückisch sind Kritikpunkte, die **plausibel** sind, aber in keiner Bewertung vorkommen. Bei Ohrhörern rechnet jeder mit Beschwerden über den Kundenservice oder die Lieferung. Ein LLM, das „typische Ohrhörer-Kritik" fortsetzt, nennt sie vielleicht, auch wenn niemand darüber geschrieben hat.

**Was passiert in der nächsten Zelle?** Wir fragen das LLM ohne unseren Themenkatalog: *„Was sind die fünf wichtigsten Kritikpunkte?"* Grundlage sind wieder die 40 Bewertungen.

In [ ]:
themen_anfrage = (f"Hier sind Bewertungen zu den Ohrhörern Nimbus Q2:\n{liste}\n\n"
                  "Nenne die fünf wichtigsten Kritikpunkte, jeweils mit 2 bis 4 Wörtern. "
                  'Antworte nur als JSON-Liste, z. B. ["Kritikpunkt 1", "Kritikpunkt 2"].')
roh = frage_llm(themen_anfrage, max_tokens=200,
                mock='["Kurze Akkulaufzeit", "App stürzt ab", "Verbindungsabbrüche", "Schlechter Kundenservice", "Hoher Preis"]')
try:
    kritikpunkte = json.loads(re.search(r"\[.*\]", roh, re.DOTALL).group(0))
except (AttributeError, json.JSONDecodeError):
    kritikpunkte = []
    print("Antwort nicht lesbar:", roh[:200])
for i, k in enumerate(kritikpunkte, 1):
    print(f"{i}. {k}")

**Wie lese ich das?** Fünf Kritikpunkte, alle plausibel. Aber stehen sie auch in den Bewertungen?

**Was passiert in der nächsten Zelle?** Für jeden Kritikpunkt suchen wir in den 40 Bewertungen nach den Wörtern daraus und zählen, wie viele Bewertungen ihn stützen. Das ist grob, aber es zeigt sofort, wo **gar nichts** zu finden ist.

In [ ]:
FUELLWOERTER = {"schlechter", "schlechte", "schlecht", "hoher", "hohe", "kurze", "kurzer", "zu", "der", "die", "das", "und", "mit", "probleme", "problem"}

def stuetzen(kritikpunkt, texte):
    woerter = [w for w in re.findall(r"\w+", kritikpunkt.lower()) if w not in FUELLWOERTER and len(w) >= 3]
    # Wortstämme: die ersten 4 Buchstaben, damit „Akkulaufzeit" auch „Akku" und „Verbindungsabbrüche" auch „Verbindung" findet
    staemme = {w[:4] for w in woerter}
    treffer = [t for t in texte if any(s in t.lower() for s in staemme)]
    return len(treffer), treffer[:2]

zeilen = []
for k in kritikpunkte:
    n, beispiele = stuetzen(k, stichprobe["text"])
    zeilen.append({"Kritikpunkt (LLM)": k, "Bewertungen, die ihn stützen": n,
                   "Beispiel": beispiele[0] if beispiele else "—",
                   "Urteil": "✅ belegt" if n >= 2 else ("⚠️ dünn" if n == 1 else "❌ nicht gefunden")})
with pd.option_context("display.max_colwidth", 80):
    display(pd.DataFrame(zeilen))

**Wie lese ich das?** Die meisten Kritikpunkte finden sich in mehreren Bewertungen wieder. Steht irgendwo ❌ oder ⚠️, lest die Bewertungen selbst: Ist der Punkt wirklich nicht da, oder findet unsere Wortsuche ihn nur nicht (andere Formulierung)? Unsere Suche kann sich irren, in beide Richtungen.

In unseren Daten gibt es **keinen** Kundenservice und **keine** Lieferung, das wissen wir aus dem Generator. Nennt das LLM so etwas, hat es einen Kritikpunkt erfunden, der zu Ohrhörern passt, nicht zu diesen Bewertungen.

Genau deshalb hat unsere Pipeline einen **festen Themenkatalog** (Sitzung 5): Das LLM darf nur aus erlaubten Themen wählen, und jedes Thema hängt an konkreten Bewertungen.

## 5. Warum lügt ein LLM?

„Lügen" ist eigentlich das falsche Wort, denn Lügen setzt Absicht voraus. Was passiert, ist einfacher, und deshalb schwerer zu verhindern:

```text
Eine Datenbank SUCHT:      Frage  ->  im Speicher nachschlagen  ->  gefunden oder „nicht vorhanden"
Ein LLM SETZT FORT:        Text   ->  wahrscheinlichstes nächstes Wort  ->  nächstes Wort  ->  ...
```

Ein LLM hat keinen Speicher mit Fakten, in dem es nachschlägt. Es erzeugt Wort für Wort die **plausibelste Fortsetzung** (Sitzung 2: Softmax über alle möglichen nächsten Wörter). Nach „Die häufigste Beschwerde ist" ist irgendein Kritikpunkt immer plausibler als „das weiß ich nicht". Eine Fortsetzung, die gut klingt, und eine, die stimmt, sind für das Modell nicht zu unterscheiden.

**Was passiert in der nächsten Zelle?** Der direkteste Test: Wir fragen nach unserem Produkt, **ohne irgendwelche Daten** mitzugeben.

In [ ]:
ohne_daten = frage_llm("Was ist die häufigste Beschwerde von Kunden über die Ohrhörer Nimbus Q2? Antworte in einem Satz.",
                       max_tokens=100, mock="Die häufigste Beschwerde betrifft die kurze Akkulaufzeit.")
print(ohne_daten)

**Wie lese ich das?** Die Nimbus Q2 gibt es nicht, wir haben sie erfunden. Das Modell kann also nichts über sie wissen. Zwei Antworten sind möglich:
- Es sagt, dass es das Produkt nicht kennt. Moderne Modelle tun das oft, weil sie darauf trainiert wurden.
- Es nennt eine **typische Ohrhörer-Beschwerde**, als wäre es eine Tatsache.

Beides ist dieselbe Mechanik: Die erste Antwort ist eine Fortsetzung, die das Training belohnt hat, keine nachgeschlagene Wahrheit. Bei einer Frage, die *ein bisschen* Kontext hat, etwa 40 Bewertungen, fällt die vorsichtige Antwort viel seltener, denn dann scheint ja Wissen da zu sein.

**Was passiert in der nächsten Zelle?** Wenn ein LLM nachschlagen würde, käme bei derselben Frage immer dieselbe Antwort. Wir stellen dieselbe Zusammenfassungs-Frage aus Abschnitt 2 **fünfmal**, diesmal mit der Standard-Temperatur, nicht mit 0.

In [ ]:
def mock_variante(i):
    rng = random.Random(i)
    return f"Rund {rng.choice([55, 60, 65, 70])} % der Kunden sind zufrieden, etwa {rng.choice([20, 25, 30, 35])} % beklagen den Akku."

laeufe = []
for i in range(5):
    t = frage_llm(roh_anfrage, system=ROH_PROMPT, temperature=None, mock=mock_variante(i))   # None = Standard, kein Cache
    laeufe.append(t)
    display(Markdown(f"**Lauf {i + 1}:** {t}"))

**Wie lese ich das?** Dieselben 40 Bewertungen, dieselbe Frage, fünf verschiedene Texte. Lest sie nebeneinander: Ändern sich nur die Formulierungen, oder auch die **Zahlen** und die **Reihenfolge der Kritikpunkte**?

**Was passiert in der nächsten Zelle?** Wir ziehen aus jedem Lauf die Prozentzahlen heraus und legen sie nebeneinander.

In [ ]:
prozente = [[int(float(z.replace(",", "."))) for z in re.findall(r"(\d+(?:[.,]\d+)?)\s*%", t)] for t in laeufe]
tabelle = pd.DataFrame({"Lauf": [f"Lauf {i + 1}" for i in range(len(laeufe))],
                        "Prozentangaben im Text": [", ".join(f"{p} %" for p in ps) or "—" for ps in prozente],
                        "Anzahl Sätze": [len(re.split(r"(?<=[.!?])\s+", t.strip())) for t in laeufe]})
display(tabelle.set_index("Lauf"))

fig, ax = plt.subplots(figsize=(8, 3))
for i, ps in enumerate(prozente):
    ax.scatter(ps, [i] * len(ps), s=60, color="#dc2626", alpha=0.7)
ax.set_yticks(range(len(laeufe))); ax.set_yticklabels([f"Lauf {i + 1}" for i in range(len(laeufe))])
ax.set_xlim(0, 100); ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0f} %"))
ax.set_title("Prozentangaben in fünf Läufen derselben Frage"); ax.invert_yaxis()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

**Wie lese ich das?** Jeder Punkt ist eine Prozentzahl aus einem Lauf. Lägen die Punkte in jeder Zeile an denselben Stellen, wären die Zahlen stabil. Meist wandern sie: mal 60 %, mal 70 %, mal gar keine Zahl.

Das ist der deutlichste Beweis, dass hier nichts **nachgeschlagen** wird. Eine Zahl, die sich bei jeder Wiederholung ändert, war nie ein Fakt. Sie war eine von mehreren plausiblen Fortsetzungen, und die Temperatur entscheidet, welche gezogen wird (Sitzung 2).

Und `temperature=0`? Dann kommt bei derselben Frage immer derselbe Text, aber nicht, weil er **richtiger** ist. Es wird nur immer dieselbe Fortsetzung gewählt. Eine erfundene Zahl wird dadurch nicht wahr, sondern nur **zuverlässig falsch**.

> 💡 **Kernpunkt:** Ein LLM setzt Text plausibel fort, es schlägt nicht nach.
>
> **Nicht:** „Das LLM weiß das, es hat die Daten gelesen."
> **Sondern:** Was nicht in den Eingaben steht und nicht geprüft wird, ist eine plausible Fortsetzung, kein Fakt.

**Was passiert in der nächsten Zelle?** Noch ein Test, diesmal mit unseren geprüften Fakten. Wir fragen nach etwas, das **nicht** in den Fakten steht: *Wie viele Kunden würden das Produkt weiterempfehlen?* Einmal ohne Regel, einmal mit der Regel „Wenn die Antwort nicht in den Fakten steht, sag das."

In [ ]:
frage = "Wie viel Prozent der Kunden würden die Nimbus Q2 weiterempfehlen? Antworte in einem Satz."
fakten_text = "Fakten:\n" + json.dumps(f_alle, ensure_ascii=False, indent=1)

ohne_regel = frage_llm(fakten_text + "\n\n" + frage, max_tokens=150,
                       mock="Etwa 60 % der Kunden würden das Produkt weiterempfehlen.")
mit_regel = frage_llm(fakten_text + "\n\n" + frage,
                      system="Beantworte Fragen nur aus den Fakten. Wenn die Antwort nicht in den Fakten steht, sag genau das.",
                      max_tokens=150, mock="Die Fakten enthalten keine Angabe zur Weiterempfehlung.")
display(Markdown(f"**Ohne Regel:** {ohne_regel}\n\n**Mit Regel:** {mit_regel}"))

**Wie lese ich das?** Weiterempfehlung steht nirgends in den Fakten. Ohne Regel leitet das Modell manchmal eine Zahl her (etwa aus dem Anteil positiver Bewertungen) und präsentiert sie, als wäre sie gemessen. Mit der Regel sagt es meistens, dass die Information fehlt.

„Meistens" ist das wichtige Wort. Die Regel verschiebt die Wahrscheinlichkeiten, sie **garantiert** nichts (Sitzung 6: Ein Prompt ist eine Bitte, kein Vertrag). Deshalb bleibt der Prüfschritt nötig, auch mit dem besten Prompt.

## 6. Fehlklassifikationen diagnostizieren

Bisher ging es um den Absatz am Ende. Aber auch die **einzelnen** Einordnungen können falsch sein, und falsche Einzelurteile wandern in jede Kennzahl. Zwei Verdächtige kennen wir aus den ersten Sitzungen: **Ironie** und **Sprache**.

**Was passiert in der nächsten Zelle?** Wie oft liegt die Pipeline bei der Stimmung richtig, getrennt nach normalen, ironischen und englischen Bewertungen?

In [ ]:
g = df[ok].copy()
g["gruppe"] = np.where(g["ironisch"], "ironisch", np.where(g["englisch"], "englisch", "normal"))
g["richtig"] = g["sentiment_llm"] == g["sentiment_wahr"]
gruppen = g.groupby("gruppe")["richtig"].agg(["mean", "sum", "count"]).reindex(["normal", "ironisch", "englisch"])

fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(gruppen.index, gruppen["mean"], color=["#9ca3af", "#dc2626", "#2563eb"])
for i, (m, n) in enumerate(zip(gruppen["mean"], gruppen["count"])):
    von, bis = radar.wilson(round(m * n), n)
    ax.plot([von, bis], [i, i], color="#111827", lw=1.5)
    ax.annotate(f"{m:.0%}  (n = {n})", (0.02, i), va="center", color="white", fontsize=9)
ax.set_xlim(0, 1); ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title("Stimmung richtig erkannt, nach Art der Bewertung"); ax.invert_yaxis()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

**Wie lese ich das?** Die schwarzen Striche sind die Intervalle aus Sitzung 3. Ironische Bewertungen sind die bekannte Schwachstelle: Positive Wörter, negative Aussage. Bei englischen Bewertungen gibt es weniger Fälle, das Intervall ist entsprechend breit.

Eine niedrige Trefferquote in einer kleinen Gruppe muss die Gesamtzahl kaum verändern. Aber sie verändert sie **in eine Richtung**: Wird Ironie als positiv gelesen, sieht das Produkt besser aus, als es ist.

**Was passiert in der nächsten Zelle?** Die ironischen Bewertungen einzeln, mit dem Urteil und der **Begründung** des Modells.

In [ ]:
ironie = g[g["ironisch"]].drop_duplicates("text")[["text", "sentiment_wahr", "sentiment_llm", "begruendung"]]
ironie["richtig"] = np.where(ironie["sentiment_llm"] == ironie["sentiment_wahr"], "✅", "❌")
with pd.option_context("display.max_colwidth", None):
    display(ironie.head(8))

**Wie lese ich das?** Bei den ❌-Zeilen lohnt die Begründung: Sie zeigt, **woran** das Modell hängen geblieben ist. Typisch: „lobt den Akku" bei „der Akku hält sagenhafte 40 Minuten". Das Modell hat die positiven Wörter gelesen, aber nicht das Missverhältnis, das die Ironie ausmacht.

Die Begründung ist damit unser wichtigstes Diagnose-Werkzeug. Ohne sie wüssten wir nur, *dass* etwas falsch ist. Mit ihr wissen wir, **welche Art** von Fehler, und ob ein besserer Prompt helfen könnte.

**Diagnose in vier Schubladen:**

```text
Ironie            -> positive Wörter, negative Aussage; Begründung zitiert das Lob
Sprache           -> englisch oder gemischt; Begründung passt zum Text, Label nicht
Wahrheit zu streng -> Modell ist vertretbar, unser Label ist diskutabel (Sitzung 1)
Modellfehler      -> Begründung passt nicht zum Text; das Modell hat etwas überlesen
```

✏️ **Eure Aufgabe 1: eine Fehlklassifikation diagnostizieren**

1. Die nächste Zelle zeigt alle Bewertungen, bei denen Pipeline und Wahrheit nicht übereinstimmen. Wählt mit `NR = ...` eine aus.
2. Lest Text, beide Labels und die Begründung.
3. Ordnet sie einer der vier Schubladen zu: `ursache = "Ironie"`, `"Sprache"`, `"Wahrheit zu streng"` oder `"Modellfehler"`, und schreibt einen Satz, warum.

Die Lösung zeigt ein Beispiel. Erst selbst versuchen.

In [ ]:
abweichend = g[~g["richtig"]].drop_duplicates("text").reset_index(drop=True)
print(f"{len(abweichend)} Bewertungen, bei denen Pipeline und Wahrheit nicht übereinstimmen.\n")

# Eure Lösung: Lücken ... füllen
# (try/except nur, damit „Alle ausführen" hier nicht stoppt. Die Lösungszelle darunter braucht es nicht.)
try:
    NR = ...                                   # eine Nummer aus der Liste, z. B. 0
    fall = abweichend.loc[int(NR)]
    print(f"Text:        {fall['text']}\nwahr:        {fall['sentiment_wahr']}\nPipeline:    {fall['sentiment_llm']}\nBegründung:  {fall['begruendung']}")
    ursache = ...                              # "Ironie", "Sprache", "Wahrheit zu streng" oder "Modellfehler"
    warum = ...                                # ein Satz
    print(f"\nDiagnose: {ursache}, weil {warum}")
except Exception as e:
    print("Noch nicht fertig oder ein Fehler:")
    print(f"  {type(e).__name__}: {e}")

In [ ]:
# Lösung (ein Beispiel): erst selbst versuchen, dann ausführen und vergleichen.
if len(abweichend):
    fall = abweichend.sort_values("ironisch", ascending=False).iloc[0]
    print(f"Text:        {fall['text']}\nwahr:        {fall['sentiment_wahr']}\nPipeline:    {fall['sentiment_llm']}\nBegründung:  {fall['begruendung']}")
    if fall["ironisch"]:
        print("\nDiagnose: Ironie, weil die Begründung die positiven Wörter aufgreift, die Aussage aber eine Beschwerde ist.")
    elif fall["englisch"]:
        print("\nDiagnose: Sprache prüfen: Passt die Begründung zum englischen Text, aber das Label nicht?")
    else:
        print("\nDiagnose: Wahrheit zu streng oder Modellfehler: Ist das Label des Modells vertretbar?")

✏️ **Eure Aufgabe 2: eine unbelegte Aussage in eurer Zusammenfassung finden**

1. Wählt einen Ausschnitt (eine Quelle oder ein Quartal), lasst die Zusammenfassung **aus Rohdaten** schreiben (wie in Abschnitt 2, mit 30 Bewertungen aus eurem Ausschnitt).
2. Lasst `satz_check` laufen, mit den Fakten **eures** Ausschnitts.
3. Sucht den Satz, der am überzeugendsten klingt und trotzdem nicht belegt ist. Notiert ihn.

In [ ]:
# Eure Lösung: Lücken ... füllen
try:
    teil = df[...]                                           # z. B. df["quelle"] == "web"
    f_teil = fakten(kennzahlen(teil), "mein Ausschnitt")
    zeilen_teil = "\n".join(f"- {t}" for t in teil.sample(min(30, len(teil)), random_state=1)["text"])
    anfrage_teil = (f"Hier sind Bewertungen zu den Ohrhörern Nimbus Q2:\n{zeilen_teil}\n\n"
                    "Fasse in 4 bis 5 Sätzen zusammen, wie zufrieden die Kunden sind und was sie am meisten stört. "
                    "Nenne dabei Prozentangaben.")
    text_teil = frage_llm(anfrage_teil, system=ROH_PROMPT, mock=mock_roh)
    display(Markdown(text_teil))
    with pd.option_context("display.max_colwidth", None):
        display(satz_check(text_teil, ...))                   # mit welchen Fakten prüfen?
except Exception as e:
    print("Noch nicht fertig oder ein Fehler:")
    print(f"  {type(e).__name__}: {e}")

In [ ]:
# Lösung: erst selbst versuchen, dann ausführen und vergleichen.
teil = df[df["quelle"] == "web"]
f_teil = fakten(kennzahlen(teil), "Nimbus Q2, Webshop")
zeilen_teil = "\n".join(f"- {t}" for t in teil.sample(min(30, len(teil)), random_state=1)["text"])
anfrage_teil = (f"Hier sind Bewertungen zu den Ohrhörern Nimbus Q2:\n{zeilen_teil}\n\n"
                "Fasse in 4 bis 5 Sätzen zusammen, wie zufrieden die Kunden sind und was sie am meisten stört. "
                "Nenne dabei Prozentangaben.")
text_teil = frage_llm(anfrage_teil, system=ROH_PROMPT, mock=mock_roh)
display(Markdown("**Zusammenfassung (Webshop, aus 30 Rohbewertungen):**\n\n" + text_teil))
with pd.option_context("display.max_colwidth", None):
    display(satz_check(text_teil, f_teil))

**Wie lese ich das?** Die Fakten kommen aus **eurem** Ausschnitt, nicht aus allen Bewertungen. Sonst würdet ihr den Text an den falschen Zahlen messen. Die überzeugendsten unbelegten Sätze sind meist die ohne Zahl: „Viele Kunden sind enttäuscht vom Akku." Sie bekommen 🔎, und erst `belege(..., teil=teil)` zeigt, ob das stimmt.

Bringt euren Satz in die Diskussion: Woran hättet ihr ihn **ohne** Werkzeug erkannt?

## 7. Gegenmittel

Was hilft gegen erfundene Aussagen? Es gibt kein einzelnes Mittel, aber eine Reihenfolge, von der stärksten Wirkung zur schwächsten:

| Gegenmittel | Wirkung | kennen wir aus |
|---|---|---|
| **Rechnen, nicht schreiben lassen:** Zahlen aus der Pipeline, das LLM formuliert nur | erfundene Zahlen werden selten | Sitzung 8 |
| **Fester Katalog** statt freier Themen | erfundene Themen werden unmöglich | Sitzung 5 |
| **Prüfen nach dem Schreiben:** Zahlen-Check, Satz-Check, Belege | Fehler werden sichtbar | heute |
| **Belege verlangen:** zu jeder Aussage die IDs der Bewertungen | Aussagen werden nachprüfbar | gleich |
| **Regeln im Prompt** („nur aus den Fakten") | hilft oft, garantiert nichts | Sitzung 6 |
| **temperature = 0** | macht Ausgaben gleich, nicht richtig | Sitzung 2 |

**Was passiert in der nächsten Zelle?** Das vierte Mittel: Wir verlangen zu jedem Kritikpunkt **zwei Bewertungs-IDs** als Beleg, und prüfen dann, ob es die IDs gibt und ob die Bewertungen das Thema wirklich nennen.

In [ ]:
mit_ids = "\n".join(f"{r}: {t}" for r, t in zip(stichprobe["review_id"], stichprobe["text"]))
beleg_anfrage = (f"Hier sind Bewertungen mit ID:\n{mit_ids}\n\n"
                 "Nenne die drei wichtigsten Kritikpunkte. Gib zu jedem genau zwei IDs von Bewertungen an, die ihn belegen. "
                 'Antworte nur als JSON-Liste: [{"kritikpunkt": "...", "thema": "<eines von: ' + ", ".join(THEMEN) + '>", "ids": ["...", "..."]}]')

def mock_belege():
    out = []
    for thema in k_alle["kritik"].index[:3]:
        ids = [r for r, l in zip(stichprobe["review_id"], stichprobe["themen_llm"]) if (thema, "negativ") in l][:2]
        out.append({"kritikpunkt": thema, "thema": thema, "ids": (ids + ["R9999"])[:2]})
    return json.dumps(out, ensure_ascii=False)

roh = frage_llm(beleg_anfrage, max_tokens=400, mock=mock_belege())
try:
    aussagen = json.loads(re.search(r"\[.*\]", roh, re.DOTALL).group(0))
except (AttributeError, json.JSONDecodeError):
    aussagen = []
    print("Antwort nicht lesbar:", roh[:200])

nach_id = stichprobe.set_index("review_id")
zeilen = []
for a in aussagen:
    for rid in a.get("ids", []):
        gibt_es = rid in nach_id.index
        nennt = gibt_es and any(t == a.get("thema") for t, _ in nach_id.loc[rid, "themen_llm"])
        zeilen.append({"Kritikpunkt": a.get("kritikpunkt"), "Thema": a.get("thema"), "ID": rid,
                       "ID existiert": "✅" if gibt_es else "❌",
                       "Bewertung nennt Thema": "✅" if nennt else "❌",
                       "Text": nach_id.loc[rid, "text"] if gibt_es else "—"})
with pd.option_context("display.max_colwidth", 70):
    display(pd.DataFrame(zeilen))

**Wie lese ich das?** Jede Zeile ist ein Beleg. Zwei Prüfungen, beide automatisch:
- **ID existiert:** Gibt es diese Bewertung überhaupt? Ein LLM kann auch IDs erfinden, die echt aussehen.
- **Bewertung nennt Thema:** Nennt die Bewertung laut unserer Pipeline das Thema, für das sie als Beleg dienen soll?

Ein ❌ in einer der beiden Spalten heißt: Dieser Beleg trägt die Aussage nicht. Das ist der große Vorteil von Belegen: Sie verwandeln „klingt plausibel" in etwas, das ein Programm **nachprüfen** kann.

> **Nicht:** „Mit einem besseren Prompt lügt das LLM nicht mehr."
> **Sondern:** Wir bauen die Pipeline so, dass jede Aussage eine prüfbare Quelle hat, und prüfen sie.

## Zusammenfassung

```text
1. Zusammenfassung aus Rohdaten      -> Lesen ist nicht Zählen: Prozente werden geschätzt oder erfunden
2. Satz für Satz prüfen               -> belegt / prüfbar / nicht belegt; Sätze ohne Zahl sind die tückischsten
3. Erfundene Themen                   -> plausible Kritik, die in keiner Bewertung steht; fester Katalog hilft
4. Mechanismus                        -> ein LLM setzt plausibel fort, es schlägt nicht nach;
                                         dieselbe Frage liefert verschiedene „Fakten"
5. temperature = 0                    -> macht Ausgaben gleich, nicht richtig
6. Fehlklassifikationen               -> Ironie, Sprache, strenge Wahrheit, Modellfehler; die Begründung ist die Spur
7. Gegenmittel                        -> rechnen statt schreiben lassen, Katalog, prüfen, Belege verlangen
```

## Zum Weiterprobieren (freiwillig)

1. **Größere Stichprobe:** Gebt dem LLM in Abschnitt 2 statt 40 dann 150 Bewertungen. Werden die Prozentzahlen genauer?
2. **Zehn Läufe:** Wiederholt Abschnitt 5 mit zehn Läufen. Welche Zahl schwankt am stärksten?
3. **Ironie im Prompt:** Ergänzt im `ANALYSE_PROMPT` einen Satz zu Ironie („Achte auf Ironie: Lob, das offensichtlich nicht ernst gemeint ist, ist Kritik"). Steigt die Trefferquote bei den ironischen Bewertungen? Und sinkt sie woanders?
4. **Belege für Lob:** Verlangt in Abschnitt 7 auch Belege für die drei wichtigsten Lobpunkte und prüft sie genauso.

## Ausblick

Heute lag der Fehler beim **Modell**. Nächste Woche (09.12) schauen wir auf die andere Seite: die **Daten**. Fake-Bewertungen, Ironie, Mehrsprachigkeit, und die Frage, wer überhaupt Bewertungen schreibt. Oft entscheidet die Datenqualität, ob ein Ergebnis stimmt, nicht die Klugheit des Modells.